[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_04/ATS_ch4_ai_robustness/ATS_ch4_ai_robustness.ipynb)

# ATS_ch4_ai_robustness

AI mini-case: how robust is the verdict of complete pass-through of ROBOR to the Romanian lending rate? The long-run coefficient of the rank-2 VECM and the p-value of the restriction across lag lengths 2-6, deterministic cases 2 and 3 and samples ending in 2019 and 2026.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 4: Cointegration revisited: VECM, ARDL and panel data. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_4'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import io

import urllib.error

from scipy import optimize

SEED = 2026

IMF_IR = 'https://api.imf.org/external/sdmx/2.1/data/IMF.STA,MFS_IR/{c}..M'

IMF_LEND = 'MFS162_RT_PT_A_PT'

IMF_DEP = 'MFS135_RT_PT_A_PT'

R3M = 'irt_st_m'

PT = {'start': '2005-08-01', 'end': '2026-03-01', 'pmax': 8, 'case': 2}

KPSW = {'start': '1949-01-01', 'end': '1988-10-01', 'pmax': 8, 'case': 3, 'H': 24}

EU27 = ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']

PANEL = {'start': 1995, 'end': 2024}

NSIM = 10000

CASES = {1: 'no deterministic terms', 2: 'restricted constant', 3: 'unrestricted constant', 4: 'restricted trend', 5: 'unrestricted trend'}

PSS_CASES = {1: 'no intercept, no trend', 2: 'restricted intercept', 3: 'unrestricted intercept', 4: 'unrestricted intercept, restricted trend', 5: 'unrestricted intercept and trend'}

_FILES = {}

_TAB = {}

HERE = '.'

def get_bytes(url):
    """Download a public file once per session (no key); optional local cache folder ATS_CACHE."""
    import hashlib
    import time
    if url in _FILES:
        return _FILES[url]
    cache = os.environ.get('ATS_CACHE')
    path = os.path.join(cache, hashlib.md5(url.encode()).hexdigest()) if cache else None
    if path and os.path.exists(path):
        _FILES[url] = open(path, 'rb').read()
        return _FILES[url]
    for attempt in range(5):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (ATS course)'})
            _FILES[url] = urllib.request.urlopen(req, timeout=300).read()
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 4:
                raise
            time.sleep(10 * (attempt + 1))
    if path:
        os.makedirs(cache, exist_ok=True)
        open(path, 'wb').write(_FILES[url])
    return _FILES[url]


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def patch(c, a=0.25):
    from matplotlib.patches import Patch
    return Patch(facecolor=c, alpha=a, edgecolor='none')


def vecm_design(Y, p, case, exog=None):
    """Z0 = dY_t, Z1 = (Y_{t-1}, restricted deterministic term), Z2 = (dY_{t-1..t-p+1}, unrestricted terms).
    Case 1: none; 2: constant in Z1; 3: constant in Z2; 4: trend in Z1, constant in Z2; 5: constant and trend in Z2.
    Y is T x n (levels); p is the lag order of the VAR in levels."""
    Y = np.asarray(Y, float)
    T, n = Y.shape
    dY = np.diff(Y, axis=0)
    t = np.arange(p, T)
    Z0 = dY[p - 1:]
    Z1 = Y[p - 1:T - 1]
    Z2 = [dY[p - 1 - j:T - 1 - j] for j in range(1, p)]
    one = np.ones((T - p, 1))
    trend = (t - p + 1.0)[:, None]
    if case == 2:
        Z1 = np.hstack([Z1, one])
    if case == 4:
        Z1 = np.hstack([Z1, trend])
    if case in (3, 4, 5):
        Z2.append(one)
    if case == 5:
        Z2.append(trend)
    if exog is not None:
        Z2.append(np.asarray(exog, float)[p:])
    Z2 = np.hstack(Z2) if Z2 else np.zeros((T - p, 0))
    return Z0, Z1, Z2


def resid(A, B):
    if B.shape[1] == 0:
        return A
    return A - B @ np.linalg.lstsq(B, A, rcond=None)[0]


def johansen(Y, p, case=3, exog=None):
    """Johansen (1988, 1991) reduced-rank regression. Returns eigenvalues (descending), beta (normalised
    beta'S11 beta = I), alpha, trace and max-eigenvalue statistics, the moment matrices and the design."""
    Z0, Z1, Z2 = vecm_design(Y, p, case, exog)
    R0, R1 = resid(Z0, Z2), resid(Z1, Z2)
    T = len(R0)
    S00, S01, S11 = R0.T @ R0 / T, R0.T @ R1 / T, R1.T @ R1 / T
    L = np.linalg.cholesky(S11)
    Li = np.linalg.inv(L)
    M = Li @ S01.T @ np.linalg.solve(S00, S01) @ Li.T
    lam, V = np.linalg.eigh((M + M.T) / 2)
    o = np.argsort(lam)[::-1]
    lam, V = np.clip(lam[o], 0, 1 - 1e-12), V[:, o]
    beta = Li.T @ V
    n = Z0.shape[1]
    lam = lam[:n]
    beta = beta[:, :n]
    alpha = S01 @ beta
    trace = np.array([-T * np.log(1 - lam[r:]).sum() for r in range(n)])
    maxeig = -T * np.log(1 - lam)
    return dict(lam=lam, beta=beta, alpha=alpha, trace=trace, maxeig=maxeig, S00=S00, S01=S01, S11=S11, T=T, n=n,
                p=p, case=case, Z0=Z0, Z1=Z1, Z2=Z2, R0=R0, R1=R1)


def vecm_fit(Y, p, r, case=3, exog=None, beta=None):
    """ML estimates of the VECM with rank r (beta from Johansen unless given): alpha, Gamma (short-run and
    unrestricted deterministic coefficients), residuals and Omega."""
    j = johansen(Y, p, case, exog)
    b = j['beta'][:, :r] if beta is None else np.asarray(beta, float).reshape(j['Z1'].shape[1], r)
    Z0, Z1, Z2 = j['Z0'], j['Z1'], j['Z2']
    ec = Z1 @ b
    X = np.hstack([ec, Z2])
    C = np.linalg.lstsq(X, Z0, rcond=None)[0]
    alpha = C[:r].T
    G = C[r:].T
    U = Z0 - X @ C
    return dict(alpha=alpha, beta=b, G=G, U=U, Omega=U.T @ U / len(U), j=j, r=r, p=p, case=case, exog=exog,
                Y=np.asarray(Y, float))


def logdet(A):
    return float(np.linalg.slogdet(A)[1])


def lr_beta(j, r, betas):
    """LR statistic of a fully specified beta (Z1-dimension x r): T[log|Omega(beta)| - log|S00| - sum log(1-lam)]."""
    S00, S01, S11, T = j['S00'], j['S01'], j['S11'], j['T']
    b = np.asarray(betas, float)
    Om = S00 - S01 @ b @ np.linalg.solve(b.T @ S11 @ b, b.T @ S01.T)
    return T * (logdet(Om) - logdet(S00) - np.log(1 - j['lam'][:r]).sum())


def partial_moments(j, A):
    """Moment matrices of the system under alpha = A psi (Johansen 1995, Ch. 8): the rows of R0 orthogonal to A are
    conditioned on. Returns S_aa.b, S_a1.b, S_11.b and log|S_bb| + the log-determinant correction."""
    n = A.shape[0]
    Abar = A @ np.linalg.inv(A.T @ A)
    if A.shape[1] == n:
        return j['S00'], j['S01'], j['S11'], 0.0
    Bp = np.linalg.svd(A.T)[2][A.shape[1]:].T            # A-perp
    Ra, Rb = j['R0'] @ Abar, j['R0'] @ Bp
    R1 = j['R1']
    T = len(Ra)
    Ra_b, R1_b = resid(Ra, Rb), resid(R1, Rb)
    Saa, Sa1, S11 = Ra_b.T @ Ra_b / T, Ra_b.T @ R1_b / T, R1_b.T @ R1_b / T
    const = logdet(Rb.T @ Rb / T) - 2 * logdet(np.hstack([Abar, Bp]))
    return Saa, Sa1, S11, const


def restricted_ml(j, r, H_list, h_list, A=None, x0=None):
    """ML under beta_i = H_i phi_i + h_i (i = 1..r; h_i carries the normalisation) and alpha = A psi, by numerical
    maximisation of the concentrated likelihood. Returns the LR statistic against the unrestricted rank-r model,
    the restricted beta and the maximised -2 log L / T (up to a constant)."""
    n = j['n']
    A = np.eye(n) if A is None else np.asarray(A, float)
    Saa, Sa1, S11, const = partial_moments(j, A)
    sizes = [H.shape[1] for H in H_list]

    def make(phi):
        cols, k = [], 0
        for H, h, s in zip(H_list, h_list, sizes):
            cols.append(H @ phi[k:k + s] + h)
            k += s
        return np.column_stack(cols)

    def f(phi):
        b = make(phi)
        BS = b.T @ S11 @ b
        if np.linalg.cond(BS) > 1e12:
            return 1e6
        return logdet(Saa - Sa1 @ b @ np.linalg.solve(BS, b.T @ Sa1.T)) + const

    k = sum(sizes)
    if k == 0:
        val, b = f(np.zeros(0)), make(np.zeros(0))
    else:
        best = None
        rng = np.random.default_rng(SEED)
        starts = [np.zeros(k) if x0 is None else np.asarray(x0, float)] + [rng.normal(0, 1, k) for _ in range(8)]
        for s0 in starts:
            res = optimize.minimize(f, s0, method='BFGS')
            if best is None or res.fun < best.fun:
                best = res
        val, b = best.fun, make(best.x)
    T = j['T']
    LR = T * (val - logdet(j['S00']) - np.log(1 - j['lam'][:r]).sum())
    return dict(LR=float(LR), beta=b, value=float(val))


def weak_exog_lr(j, r, A):
    """LR test of alpha = A psi with beta unrestricted: T sum log((1 - lam_tilde_i)/(1 - lam_i)) (Johansen 1995)."""
    Saa, Sa1, S11, _ = partial_moments(j, np.asarray(A, float))
    L = np.linalg.cholesky(S11)
    Li = np.linalg.inv(L)
    M = Li @ Sa1.T @ np.linalg.solve(Saa, Sa1) @ Li.T
    lt = np.sort(np.linalg.eigvalsh((M + M.T) / 2))[::-1][:r]
    return float(j['T'] * np.log((1 - lt) / (1 - j['lam'][:r])).sum())


def var_from_vecm(alpha, beta_y, G, p, n):
    """Levels VAR(p) slope matrices from the VECM: A1 = I + alpha beta' + Gamma1, Ai = Gamma_i - Gamma_{i-1},
    Ap = -Gamma_{p-1}. beta_y: the rows of beta that multiply Y_{t-1}."""
    Gam = [G[:, n * i:n * (i + 1)] for i in range(p - 1)]
    Pi = alpha @ beta_y.T
    A = []
    for i in range(p):
        Ai = (np.eye(n) + Pi if i == 0 else np.zeros((n, n)))
        if i < p - 1:
            Ai = Ai + Gam[i]
        if i > 0:
            Ai = Ai - Gam[i - 1]
        A.append(Ai)
    return np.stack(A)


def ma_coefs(A, H):
    """Reduced-form MA coefficients Phi_0..Phi_H (Phi_0 = I) of a VAR in levels (need not be stable)."""
    p, n, _ = A.shape
    Phi = np.zeros((H + 1, n, n))
    Phi[0] = np.eye(n)
    for h in range(1, H + 1):
        Phi[h] = sum(A[j] @ Phi[h - j - 1] for j in range(min(p, h)))
    return Phi


def perp(M):
    """Orthogonal complement of the columns of M (n x r) -> n x (n - r)."""
    u, s, vt = np.linalg.svd(M)
    return u[:, M.shape[1]:]


def granger_C(alpha, beta_y, G, p, n):
    """Long-run impact matrix of the Granger representation: C = beta_perp (alpha_perp' Gamma beta_perp)^{-1}
    alpha_perp', Gamma = I - sum Gamma_i."""
    Gam = np.eye(n) - sum(G[:, n * i:n * (i + 1)] for i in range(p - 1)) if p > 1 else np.eye(n)
    ap, bp = perp(alpha), perp(beta_y)
    return bp @ np.linalg.solve(ap.T @ Gam @ bp, ap.T)


def lag_select(Y, pmax, case=3, exog=None):
    """AIC, BIC and HQ of VARs in levels with p = 1..pmax on a common sample (constant included)."""
    Y = np.asarray(Y, float)
    out = {}
    T, n = Y.shape
    for p in range(1, pmax + 1):
        Yt = Y[pmax:]
        X = np.hstack([Y[pmax - j:T - j] for j in range(1, p + 1)] + [np.ones((T - pmax, 1))])
        U = Yt - X @ np.linalg.lstsq(X, Yt, rcond=None)[0]
        S = U.T @ U / len(U)
        k = X.shape[1] * n
        ld = logdet(S)
        out[p] = dict(aic=ld + 2 * k / len(U), bic=ld + np.log(len(U)) * k / len(U),
                      hq=ld + 2 * np.log(np.log(len(U))) * k / len(U))
    return {c: int(min(out, key=lambda q: out[q][c])) for c in ('aic', 'bic', 'hq')}


def sim_trace_null(m, case, T=400, reps=NSIM, seed=SEED):
    """Null distribution of the trace statistic for H(r) against H(n) when n - r = m, simulated from m independent
    random walks (with a drift in cases 3 and 5, the configuration for which these cases are tabulated) and a VAR(1)
    test regression. T = 400 approximates the asymptotic distribution."""
    rng = np.random.default_rng(seed + 100 * case + m)
    out = np.empty(reps)
    for k in range(reps):
        e = rng.standard_normal((T, m))
        if case == 3:
            e = e + 1.0
        if case == 5:
            e = e + 1.0 + 0.02 * np.arange(T)[:, None]
        Y = np.cumsum(e, axis=0)
        out[k] = johansen(Y, 1, case)['trace'][0]
    return out


def trace_tables(ms=(1, 2, 3, 4), reps=NSIM):
    """90%, 95% and 99% quantiles of the simulated trace distributions, cases 1-5, n - r = 1..4."""
    tab = {}
    draws = {}
    for case in range(1, 6):
        for m in ms:
            d = sim_trace_null(m, case, reps=reps)
            draws[(case, m)] = d
            tab[f'{case}|{m}'] = [float(np.quantile(d, q)) for q in (0.90, 0.95, 0.99)]
    return tab, draws


def crit_table():
    """The simulated 90/95/99% quantiles of the trace statistic (cases 1-5, n - r = 1..4), computed once per session
    (or read from ch4_numbers.json when it exists, so that every slide uses the same table)."""
    if not _TAB:
        path = os.path.join(HERE, 'ch4_numbers.json') if 'HERE' in globals() else ''
        if path and os.path.exists(path) and 'trace' in json.load(open(path)):
            _TAB.update(json.load(open(path))['trace']['table'])
        else:
            _TAB.update(trace_tables(reps=NSIM)[0])
    return _TAB


def pvalue(stat, draws):
    return float((np.sum(draws >= stat) + 1) / (len(draws) + 1))


def ols(y, X):
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    u = y - X @ b
    s2 = u @ u / (len(y) - X.shape[1])
    V = s2 * np.linalg.inv(X.T @ X)
    return b, V, u


def ardl_ecm(y, x, p, q, case=3, start=None):
    """Conditional ECM of an ARDL(p, q) (PSS 2001, eq. 6):
    dy_t = det + pi_y y_{t-1} + pi_x' x_{t-1} + sum_{i<p} psi_i dy_{t-i} + sum_{j<q} omega_j' dx_{t-j} + u_t.
    y: (T,), x: (T, k). Returns F (bounds F statistic), t (t-ratio of pi_y), the long-run coefficients
    theta = -pi_x/pi_y with delta-method standard errors, the speed of adjustment pi_y, information criteria and the
    fitted pieces. `start` fixes the first usable observation (common sample for lag selection)."""
    y = np.asarray(y, float)
    x = np.asarray(x, float).reshape(len(y), -1)
    T, k = x.shape
    m = max(p, q, 1)
    s = start if start is not None else m
    dy, dx = np.diff(y, prepend=np.nan), np.diff(x, axis=0, prepend=np.nan)
    rows = np.arange(s, T)
    cols, names = [], []
    tt = rows.astype(float)
    if case == 2:
        cols.append(np.ones(len(rows)))
        names.append('c_r')
    if case in (3, 4, 5):
        cols.append(np.ones(len(rows)))
        names.append('c')
    if case == 4:
        cols.append(tt)
        names.append('t_r')
    if case == 5:
        cols.append(tt)
        names.append('t')
    cols.append(y[rows - 1])
    names.append('y1')
    for i in range(k):
        cols.append(x[rows - 1, i])
        names.append(f'x1_{i}')
    for i in range(1, p):
        cols.append(dy[rows - i])
        names.append(f'dy{i}')
    for i in range(k):
        for j in range(q):
            cols.append(dx[rows - j, i])
            names.append(f'dx{j}_{i}')
    X = np.column_stack(cols)
    b, V, u = ols(dy[rows], X)
    test = [nm for nm in names if nm in ('c_r', 't_r', 'y1') or nm.startswith('x1_')]
    idx = [names.index(nm) for nm in test]
    R = np.zeros((len(idx), len(b)))
    R[np.arange(len(idx)), idx] = 1
    F = float((R @ b) @ np.linalg.solve(R @ V @ R.T, R @ b) / len(idx))
    iy = names.index('y1')
    ix = [names.index(f'x1_{i}') for i in range(k)]
    piy = b[iy]
    theta = -b[ix] / piy
    se_th = []
    for i in ix:
        g = np.zeros(len(b))
        g[i] = -1 / piy
        g[iy] = b[i] / piy ** 2
        se_th.append(float(np.sqrt(g @ V @ g)))
    n = len(u)
    sig2 = u @ u / n
    return dict(F=F, t=float(piy / np.sqrt(V[iy, iy])), theta=theta, se_theta=np.array(se_th), phi=float(piy),
                se_phi=float(np.sqrt(V[iy, iy])), aic=float(np.log(sig2) + 2 * len(b) / n),
                bic=float(np.log(sig2) + np.log(n) * len(b) / n), b=b, V=V, u=u, names=names, n=n, p=p, q=q, case=case,
                halflife=float(np.log(0.5) / np.log(1 + piy)) if -1 < piy < 0 else np.nan)


def ardl_select(y, x, pmax=6, qmax=6, case=3, crit='aic'):
    """Lag orders by AIC or BIC on a common sample (start = max lag)."""
    m = max(pmax, qmax)
    best = None
    for p in range(1, pmax + 1):
        for q in range(1, qmax + 1):
            r = ardl_ecm(y, x, p, q, case, start=m)
            if best is None or r[crit] < best[crit]:
                best = r
    return ardl_ecm(y, x, best['p'], best['q'], case)


def bounds_sim(k, case, T=1000, reps=NSIM, seed=SEED, which='F'):
    """Null distribution of the bounds F (or t) statistic: y a random walk unrelated to x; x either k independent
    random walks (the I(1) bound) or k i.i.d. N(0,1) series (the I(0) bound). Regression dy_t on the deterministic
    terms of the case, y_{t-1}, x_{t-1} (short-run terms do not affect the limit). Returns (I(0) draws, I(1) draws)."""
    rng = np.random.default_rng(seed + 10 * case + k + T)
    out = {0: np.empty(reps), 1: np.empty(reps)}
    for bnd in (0, 1):
        for r in range(reps):
            y = np.cumsum(rng.standard_normal(T + 1))
            e = rng.standard_normal((T + 1, k))
            x = np.cumsum(e, axis=0) if bnd == 1 else e
            out[bnd][r] = ardl_ecm(y, x, 1, 0, case)[which]
    return out[0], out[1]


def bounds_quantiles(k, case, T=1000, reps=NSIM, which='F'):
    lo, hi = bounds_sim(k, case, T, reps, which=which)
    qs = (0.90, 0.95, 0.99) if which == 'F' else (0.10, 0.05, 0.01)
    return {f'{q:.2f}': [float(np.quantile(lo, q)), float(np.quantile(hi, q))] for q in qs}


def reinsel_ahn(trace, T, n, p):
    """Reinsel and Ahn (1992): the trace statistic scaled by (T - n p)/T."""
    return trace * (T - n * p) / T


def vecm_simulate(m, e):
    """Levels generated recursively from a fitted VECM with innovations e (T_eff x n), initial values and
    deterministic terms taken from the estimation sample."""
    Y, j, n, p = m['Y'], m['j'], m['j']['n'], m['p']
    Z1d, Z2d = j['Z1'][:, n:], j['Z2'][:, n * (p - 1):]
    T = len(e)
    Yb = np.zeros((T + p, n))
    Yb[:p] = Y[:p]
    for k in range(T):
        t = p + k
        z1 = np.concatenate([Yb[t - 1], Z1d[k]])
        dyl = [Yb[t - i] - Yb[t - i - 1] for i in range(1, p)]
        z2 = np.concatenate(dyl + [Z2d[k]]) if dyl else Z2d[k]
        Yb[t] = Yb[t - 1] + m['alpha'] @ (m['beta'].T @ z1) + m['G'] @ z2 + e[k]
    return Yb


def boot_trace(Y, p, r, case=2, B=399, seed=SEED, wild=True, exog=None):
    """Bootstrap p-value of the trace test of H(r) (Swensen 2006; wild bootstrap of Cavaliere, Rahbek and Taylor
    2010, 2012): the VECM is estimated under H(r), data are regenerated recursively with residuals multiplied by
    N(0, 1) draws (wild) or resampled (i.i.d.), and the trace statistic is recomputed on each bootstrap sample."""
    j = johansen(Y, p, case, exog)
    stat = j['trace'][r]
    m = vecm_fit(Y, p, r, case, exog)
    rng = np.random.default_rng(seed)
    U = m['U']
    T = len(U)
    cnt, done = 0, 0
    for _ in range(B):
        e = U * rng.standard_normal((T, 1)) if wild else (U - U.mean(0))[rng.integers(0, T, T)]
        Yb = vecm_simulate(m, e)
        try:
            sb = johansen(Yb, p, case, exog)['trace'][r]
        except np.linalg.LinAlgError:
            continue
        cnt += sb >= stat
        done += 1
    return float(stat), float((cnt + 1) / (done + 1))


def read_imf_rates(country):
    """Monthly interest rates of one country from the IMF data API (International Financial Statistics, dataset
    MFS_IR; public SDMX, no key). country: ISO3 code, e.g. 'ROU'. Returns a DataFrame, one column per indicator."""
    xml = get_bytes(IMF_IR.format(c=country)).decode('utf-8', 'ignore')
    out = {}
    for m in re.finditer(r'<Series ([^>]*)>(.*?)</Series>', xml, re.S):
        ind = re.search(r'INDICATOR="(\w+)"', m.group(1)).group(1)
        obs = re.findall(r'TIME_PERIOD="(\d{4})-M(\d{2})" OBS_VALUE="([^"]+)"', m.group(2))
        if obs:
            out[ind] = pd.Series({pd.Timestamp(int(y), int(mm), 1): float(v) for y, mm, v in obs}).sort_index()
    return pd.DataFrame(out)


def passthrough_data(geo='RO', iso='ROU', start=PT['start'], end=PT['end']):
    """Lending rate and deposit rate (IMF IFS, national-currency business, % p.a.) and the 3-month money-market rate
    (Eurostat irt_st_m: ROBOR, PRIBOR, BUBOR), monthly. Isolated missing months of the money-market rate (BUBOR has
    19 in 2006-2013) are filled by linear interpolation (at most two consecutive months)."""
    imf = read_imf_rates(iso)
    r3 = read_eurostat(R3M, f'M.IRT_M3.{geo}')
    r3 = r3.reindex(pd.date_range(r3.index[0], r3.index[-1], freq='MS')).interpolate(limit=2, limit_area='inside')
    d = pd.concat([imf[IMF_LEND], imf[IMF_DEP], r3], axis=1, keys=['lend', 'dep', 'mm']).loc[start:end]
    return d.dropna()


def passthrough_tests(d, p, case=2):
    """Rank-2 pass-through VECM (lending, deposit, money-market rate): just-identified beta (lending excludes the
    deposit rate, deposit excludes the lending rate), LR tests of complete pass-through and weak exogeneity."""
    Y = d[['lend', 'dep', 'mm']].values
    j = johansen(Y, p, case)
    nz = j['Z1'].shape[1]
    free = np.zeros((nz, 2 if case == 2 else 1))
    free[2, 0] = 1
    if case == 2:
        free[3, 1] = 1
    e = np.eye(nz)
    ji = restricted_ml(j, 2, [free, free], [e[0], e[1]])
    fixed = np.zeros((nz, 1 if case == 2 else 0))
    if case == 2:
        fixed[3, 0] = 1
    jc = restricted_ml(j, 2, [fixed, fixed], [e[0] - e[2], e[1] - e[2]])
    jl = restricted_ml(j, 2, [fixed, free], [e[0] - e[2], e[1]])
    we = weak_exog_lr(j, 2, np.array([[1, 0], [0, 1], [0, 0.0]]))
    b = ji['beta']
    m = vecm_fit(Y, p, 2, case, beta=b)
    return dict(j=j, beta=b, alpha=m['alpha'], LRc=jc['LR'], pc=float(1 - stats.chi2.cdf(jc['LR'], 2)),
                LRl=jl['LR'], pl=float(1 - stats.chi2.cdf(jl['LR'], 1)), LRw=we, pw=float(1 - stats.chi2.cdf(we, 2)),
                theta_l=float(-b[2, 0]), theta_d=float(-b[2, 1]), mk_l=float(-b[3, 0]) if case == 2 else np.nan,
                mk_d=float(-b[3, 1]) if case == 2 else np.nan, m=m)


def fig_ai_case(save_it=True):
    """Robustness of one verdict: the long-run pass-through of ROBOR to the lending rate and the p-value of complete
    pass-through, across lag lengths, deterministic cases and samples."""
    rows = []
    for end, lab in (('2019-12-01', 'to 2019'), (PT['end'], 'to 2026')):
        d = passthrough_data(end=end)
        for case in (2, 3):
            for p in (2, 3, 4, 5, 6):
                r = passthrough_tests(d, p, case)
                rows.append(dict(sample=lab, case=case, p=p, theta=r['theta_l'], pl=r['pl']))
    t = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(11, 3.9))
    xs = {p: i for i, p in enumerate((2, 3, 4, 5, 6))}
    for (lab, case), c, mk, off in ((('to 2026', 2), st.MainBlue, 'o', -0.15), (('to 2026', 3), st.Teal, 's', -0.05),
                                    (('to 2019', 2), st.IDAred, 'o', 0.05), (('to 2019', 3), st.Orange, 's', 0.15)):
        s = t[(t['sample'] == lab) & (t['case'] == case)]
        x = np.array([xs[p] for p in s['p']]) + off
        filled = s['pl'].values < 0.05
        ax.scatter(x[filled], s['theta'].values[filled], color=c, marker=mk, s=48, zorder=3,
                   label=f'case {case}, sample {lab}')
        ax.scatter(x[~filled], s['theta'].values[~filled], facecolors='none', edgecolors=c, marker=mk, s=48, zorder=3)
    ax.axhline(1, color=st.DarkText, lw=0.8, ls='--')
    ax.set_xticks(list(xs.values()))
    ax.set_xticklabels([f'p = {p}' for p in xs])
    ax.set_ylabel('long-run pass-through')
    ax.set_xlabel('lag order of the VAR (filled marker: complete pass-through rejected at 5%)')
    st.legend_outside_bottom(ax, ncol=4, y=-0.2)
    save('ats_ch4_ai_case', save_it)
    return dict(rows=rows, tmin=float(t['theta'].min()), tmax=float(t['theta'].max()), nrej=int((t['pl'] < 0.05).sum()),
                n=int(len(t)))

## Run

In [ ]:
print(fig_ai_case())

![ats_ch4_ai_case](./ats_ch4_ai_case.png)

Output: `ats_ch4_ai_case.pdf`